In [ ]:
import os
import random

from toolz import *
from toolz.curried import *
from toolz.curried.operator import *

from pathlib import Path

import pandas as pd
import numpy as np

from ast import literal_eval as eval

import embed_toolkit

import pydicom
import mapply

In [ ]:
# NOTE : Please set BASE_DIR to the path where you have downloaded the embed data
LOAD_DIR: Path = Path(os.path.abspath("[where/you/have/downloaded/embed/data]/"))
SAVE_DIR: Path = Path(os.path.abspath("../processed/embed/"))

# NOTE : Please ensure that the following files are present in BASE_DIR/tables/
# NOTE : the file names for metadata and magview files may be different in your setup
METADATA_PATH: Path = LOAD_DIR / "tables/meta.csv"
MAGVIEW_PATH: Path = LOAD_DIR / "tables/magview.csv"

MASS_COLUMNS: list[str] = ["massshape", "massmargin", "massdens"]
CALC_COLUMNS: list[str] = ["calcdistri", "calcfind", "calcnumber"]

MASS = "mass"
CALC = "calcification"
AD = "architectural_distortion"
AS = "asymmetry"

embed_params = embed_toolkit.EMBEDParameters()

In [ ]:
# load csv files
magview_df = pd.read_csv(
    MAGVIEW_PATH,
    usecols=[
        "empi_anon",
        "acc_anon",
        "numfind",
        "side",
        "asses",
        "desc",
        "path_severity",
    ]
    + MASS_COLUMNS
    + CALC_COLUMNS,
)

meta_df = pd.read_csv(
    METADATA_PATH,
    usecols=[
        "empi_anon",
        "acc_anon",
        "ImageLateralityFinal",
        "ViewPosition",
        "FinalImageType",
        "anon_dicom_path",
        "ROI_coords",
        "ProtocolName",
        "Rows",
        "Columns",
    ],
)

# rename columns for my own convenience
magview_df = magview_df.rename(
    columns=merge(
        {
            "empi_anon": "patient_id",
            "acc_anon": "study_id",
            "numfind": "num_findings",
            "side": "laterality",
            "asses": "birads",
            "desc": "description",
            "path_severity": "path_severity",
        },
        {col: col for col in MASS_COLUMNS},
        {col: col for col in CALC_COLUMNS},
    ),
)

meta_df = meta_df.rename(
    columns={
        "empi_anon": "patient_id",
        "acc_anon": "study_id",
        "ImageLateralityFinal": "laterality",
        "ViewPosition": "view_position",
        "FinalImageType": "image_type",
        "anon_dicom_path": "dcm_path",
        "ROI_coords": "rois",
        "ProtocolName": "protocol_name",
        "Rows": "dcm_height",
        "Columns": "dcm_width",
    },
)

In [ ]:
# extract only the screenings
def df_contains(case):
    return lambda series: series.str.lower().str.contains(case, na=False)


magview_df = magview_df[df_contains("screen")(magview_df["description"])]

# extract findings with mass, calcification, asymmetry, or architectural distortion
magview_df[["mass", "asymmetry", "arch_distortion", "calcification"]] = magview_df.apply(
    embed_params.extract_characteristics, axis="columns", result_type="expand"
)

# single finding only
magview_df = magview_df[magview_df["num_findings"] == 1]

magview_df = (
    magview_df.groupby(["patient_id", "study_id", "laterality"], group_keys=False)
    .apply(lambda x: x.sort_values("path_severity").head(1))
    .reset_index(drop=True)
)


# extract pathology-based severity
def severity_fn(row):
    if row["path_severity"] in [0, 1]:
        return 1
    else:
        return 0


magview_df["path_severity"] = magview_df.apply(severity_fn, axis="columns")

In [ ]:
# only combo have spatially matching Mammo-DBT pairs
meta_df = meta_df[df_contains("combohd")(meta_df["protocol_name"])]

# get only 2D and 3D, not c-view
meta_df = meta_df[meta_df["image_type"].isin(["2D", "3D"])]

# extract mammo
meta_2d_df = meta_df[meta_df["image_type"] == "2D"]  # only 2D
meta_2d_df = meta_2d_df[(meta_2d_df["rois"] != "[]")]  # remove empty roi
meta_2d_df["rois"] = meta_2d_df["rois"].apply(eval)  # convert str to list
meta_2d_df = meta_2d_df[meta_2d_df["rois"].apply(compose(eq(1), len))]
# remove recall
meta_2d_df = meta_2d_df.groupby(["patient_id", "study_id", "laterality", "view_position"]).filter(lambda x: len(x) == 1)

# extract dbt
meta_3d_df = meta_df[meta_df["image_type"] == "3D"]
# remove recall
meta_3d_df = meta_3d_df.groupby(["patient_id", "study_id", "laterality", "view_position"]).filter(lambda x: len(x) == 1)

In [ ]:
# join
meta_df = pd.merge(
    meta_2d_df,
    meta_3d_df,
    how="inner",
    on=["patient_id", "study_id", "laterality", "view_position"],
    suffixes=("_mammo", "_dbt"),
)

# select the desired columns
meta_df = meta_df[
    [
        "patient_id",
        "study_id",
        "laterality",
        "view_position",
        "dcm_path_mammo",
        "dcm_path_dbt",
        "rois_mammo",
        "protocol_name_dbt",
    ]
]

# remove nan dcm paths
meta_df = meta_df[~meta_df["dcm_path_dbt"].isna()]

# rename for my own convenience
meta_df = meta_df.rename(
    columns={
        "patient_id": "patient_id",
        "study_id": "study_id",
        "laterality": "laterality",
        "view_position": "view_position",
        "dcm_path_mammo": "dcm_path_mammo",
        "dcm_path_dbt": "dcm_path_dbt",
        "rois_mammo": "rois",
        "protocol_name_dbt": "protocol_name",
    }
)

In [ ]:
# merge it
merged_df = pd.merge(
    magview_df,
    meta_df,
    how="left",
    left_on=["patient_id", "study_id", "laterality"],
    right_on=["patient_id", "study_id", "laterality"],
)

# final touch
merged_df = merged_df[
    [
        "patient_id",
        "study_id",
        "dcm_path_mammo",
        "dcm_path_dbt",
        "laterality",
        "view_position",
        "rois",
        "mass",
        "calcification",
        "arch_distortion",
        "asymmetry",
        "path_severity",
        "protocol_name",
    ]
]

merged_df = merged_df[~merged_df["rois"].isna()]

In [ ]:
# Get DICOM image metadata
class DCM_Tags:
    def __init__(self, img_dcm):
        try:
            self.laterality = img_dcm.ImageLaterality
        except AttributeError:
            self.laterality = np.nan

        try:
            self.view = img_dcm.ViewPosition
        except AttributeError:
            self.view = np.nan

        try:
            self.orientation = img_dcm.PatientOrientation
        except AttributeError:
            self.orientation = np.nan


# Check whether DICOM should be flipped
def get_flip_rule(imgdcm):

    # Get DICOM metadata
    tags = DCM_Tags(imgdcm)

    # If image orientation tag is defined
    if ~pd.isnull(tags.orientation):
        # CC view
        if tags.view == "CC":
            if tags.orientation[0] == "P":
                flipHorz = True
            else:
                flipHorz = False

            if (tags.laterality == "L") & (tags.orientation[1] == "L"):
                flipVert = True
            elif (tags.laterality == "R") & (tags.orientation[1] == "R"):
                flipVert = True
            else:
                flipVert = False

        # MLO or ML views
        elif (tags.view == "MLO") | (tags.view == "ML"):
            if tags.orientation[0] == "P":
                flipHorz = True
            else:
                flipHorz = False

            if (tags.laterality == "L") & ((tags.orientation[1] == "H") | (tags.orientation[1] == "HL")):
                flipVert = True
            elif (tags.laterality == "R") & ((tags.orientation[1] == "H") | (tags.orientation[1] == "HR")):
                flipVert = True
            else:
                flipVert = False

        # Unrecognized view
        else:
            flipHorz = False
            flipVert = False

    # If image orientation tag is undefined
    else:
        # Flip RCC, RML, and RMLO images
        if (tags.laterality == "R") & ((tags.view == "CC") | (tags.view == "ML") | (tags.view == "MLO")):
            flipHorz = True
            flipVert = False
        else:
            flipHorz = False
            flipVert = False

    return flipHorz, flipVert


def flip_roi(roi, image_shape, flipHorz, flipVert):

    if roi and len(roi) == 4:
        # roi: [ymin, xmin, ymax, xmax]
        H, W = image_shape

        ymin, xmin, ymax, xmax = roi

        if flipHorz:
            xmin, xmax = W - xmax, W - xmin
        if flipVert:
            ymin, ymax = H - ymax, H - ymin
        return [ymin, xmin, ymax, xmax]
    else:
        return roi


processed_df = merged_df.copy()

# Add a column with a random unique number for each row
random.seed(0)
processed_df["patch_id"] = [random.getrandbits(25) for _ in range(len(processed_df))]


def update_roi(row):
    dcm_path_mammo = row["dcm_path_mammo"]
    dcm = pydicom.dcmread(dcm_path_mammo)
    array = dcm.pixel_array
    h, w = array.shape
    flipped_roi = flip_roi(row["rois"][0], array.shape, *get_flip_rule(dcm))
    return [flipped_roi, h, w]


# flip roi
mapply.init(n_workers=64)
processed_df[["flipped_rois", "H", "W"]] = processed_df.mapply(update_roi, axis=1, result_type="expand")

processed_df = processed_df.rename(columns={"path_severity": "severity"})

processed_df.to_csv(SAVE_DIR / "meta.csv", index=False)